##Qwen2.5-1.5B fine-tuning with LoRA for SI



In [ ]:
from pathlib import Path

base_path = Path("SemEval-2020/datasets")

articles_path = base_path / "train-articles"
si_labels_path = base_path / "train-labels-task1-span-identification"

articles = {}
for txt in articles_path.glob("*.txt"):
    article_id = txt.stem.replace("article", "")
    articles[article_id] = txt.read_text(encoding="utf-8")

si_labels = {}
for label in si_labels_path.glob("*.labels"):
    article_id = label.stem.replace("article", "").replace(".task1-SI", "")
    spans = []
    for line in label.read_text(encoding="utf-8").splitlines():
        _, start, end = line.split("\t")
        spans.append((int(start), int(end)))
    si_labels[article_id] = spans

In [ ]:
import re

token_pattern = re.compile(r"\w+|[^\w\s]")

def tokenize_and_label(text, spans):
    tokens, labels = [], []

    for match in re.finditer(token_pattern, text):
        token = match.group()
        start, end = match.span()

        label = "O"
        for s, e in spans:
            if start >= s and start < e:
                label = "B" if start == s else "I"
                break

        tokens.append(token)
        labels.append(label)

    return tokens, labels

In [ ]:
from datasets import Dataset

data = []
for article_id, text in articles.items():
    spans = si_labels.get(article_id, [])
    tokens, labels = tokenize_and_label(text, spans)
    data.append({"tokens": tokens, "labels": labels})

dataset = Dataset.from_list(data)

In [ ]:
print(dataset[5])

In [ ]:
!uv pip install transformers==4.56.2

Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 285ms
Prepared 2 packages in 584ms
Uninstalled 2 packages in 516ms
Installed 2 packages in 56ms
 - huggingface-hub==1.8.0
 + huggingface-hub==0.36.2
 - transformers==5.0.0
 + transformers==4.56.2


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorWithPadding, TrainerCallback,
    EvalPrediction,
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def sample_to_windows(sample, stride = 121, max_input_tokens = 128):
  tokens, labels = sample["tokens"], sample["labels"]
  n = len(tokens)
  windows = []
  start = 0
  while start < n:
    end = min(start + max_input_tokens, n)
    windows.append((tokens[start:end], labels[start:end]))
    if end == n:
        break
    start += stride
  return windows

In [ ]:
def tokenize(chunk_tokens, chunk_labels, max_length = 512):
  messages_prompt = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user",   "content": f"Tokens: {' '.join(chunk_tokens)}"},
  ]
  messages_full = messages_prompt + [
      {"role": "assistant", "content": " ".join(chunk_labels)}
  ]
  prompt_text = tokenizer.apply_chat_template(
      messages_prompt,
      tokenize=False,
      add_generation_prompt=True
      )
  full_text = tokenizer.apply_chat_template(
      messages_full,
      tokenize=False,
      add_generation_prompt=False
      )
  prompt_ids = tokenizer(prompt_text,
                         add_special_tokens=False
                         )["input_ids"]

  full_enc   = tokenizer(full_text,
                         add_special_tokens=False,
                         max_length=max_length,
                         truncation=True)
  input_ids  = full_enc["input_ids"]

  labels_ids = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]

  return {"input_ids": input_ids,
          "attention_mask": full_enc["attention_mask"],
          "labels": labels_ids}

In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
eval_data = split["test"]

In [ ]:
SYSTEM_PROMPT = """You are a propaganda detection assistant.
Your task is to perform token-level labeling using the BIO format to detect propaganda.
- Label each token as B if it is the beginning of a propaganda span.
- Label as I if it is inside the span.
- Label as O if it is outside.
Return ONLY a list of labels in the same order as the tokens, separated by spaces."""

In [ ]:
train_dataset = Dataset.from_list([tokenize(ct, cl)
                              for s in train_data
                              for ct, cl in sample_to_windows(s)])

eval_dataset  = Dataset.from_list([tokenize(ct, cl)
                              for s in eval_data
                              for ct, cl in sample_to_windows(s)])

In [ ]:
print(f"Original samples train: {len(train_data)} eval: {len(eval_data)}")
print(f"After windowing train: {len(train_dataset)} eval: {len(eval_dataset)}")

Original samples train: 333 eval: 38
After windowing train: 3774 eval: 435


In [ ]:
for k, v in train_dataset[0].items():
  print(f"{k}: {v}")

###zero-shot

In [ ]:
def predict_with_qwen(model, sample, stride = 121, max_input_tokens = 128):
  tokens = sample["tokens"]
  n = len(tokens)
  valid = {"B", "I", "O"}
  final_labels = ["O"] * n

  windows   = sample_to_windows(sample)
  win_start = 0
  for w_idx, (chunk_tokens, _) in enumerate(tqdm(windows, desc="Sample windows", leave=False)):
    w_len = len(chunk_tokens)
    is_last = (w_idx == len(windows) - 1)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Tokens: {' '.join(chunk_tokens)}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt,
                       return_tensors="pt",
                       padding=True
                       ).to(device)

    with torch.no_grad():
      out = model.generate(
          **inputs,
          max_new_tokens = w_len,
          # max_new_tokens = w_len * 3,
          do_sample = False,
          pad_token_id = tokenizer.eos_token_id,
      )

    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
        ).strip().split()

    decoded = (decoded + ["O"] * w_len)[:w_len]
    decoded = [t if t in valid else "O" for t in decoded]

    overlap = max_input_tokens - stride
    if is_last:
        write_start = win_start
        write_end = win_start + w_len
        src_slice = decoded[:w_len]
    else:
        write_start = win_start if w_idx == 0 else win_start + overlap
        write_end = win_start + w_len
        src_start = 0 if w_idx == 0 else overlap
        src_slice = decoded[src_start:w_len]

    final_labels[write_start:write_end] = src_slice

    if w_idx == 0:
        win_start += w_len
    else:
        win_start += stride

  return final_labels[:n]

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
from tqdm.notebook import tqdm
model.eval()
true_labels, pred_labels = [], []

for sample in tqdm(eval_data):
  pred = predict_with_qwen(model, sample)
  true_labels.extend(sample["labels"])
  pred_labels.extend(pred)

In [ ]:
print("ZERO-SHOT EVALUATION")
print(classification_report(true_labels, pred_labels, labels=["B", "I", "O"], zero_division=0))

zeroshot_metrics = {
    "accuracy": accuracy_score(true_labels, pred_labels),
    "P": precision_score(true_labels, pred_labels, average="macro", zero_division=0),
    "R":  recall_score(true_labels, pred_labels, average="macro", zero_division=0),
    "F1": f1_score(true_labels, pred_labels, average="macro", zero_division=0)}

print(
    f"accuracy {zeroshot_metrics['accuracy'] * 100:.0f}%, "
    f"F1 {zeroshot_metrics['F1'] * 100:.0f}%, "
    f"P {zeroshot_metrics['P'] * 100:.0f}%, "
    f"R {zeroshot_metrics['R'] * 100:.0f}%"
)

ZERO-SHOT EVALUATION
              precision    recall  f1-score   support

           B       0.00      0.00      0.00       511
           I       0.13      0.23      0.16      5255
           O       0.87      0.78      0.82     36749

    accuracy                           0.70     42515
   macro avg       0.33      0.34      0.33     42515
weighted avg       0.76      0.70      0.73     42515

accuracy 70%, F1 33%, P 33%, R 34%


In [ ]:
import json

with open("zeroshot_metrics.json", "w") as f:
  json.dump(zeroshot_metrics, f, indent=4)

In [ ]:
del model
torch.cuda.empty_cache()

###Fine-tuning with LoRA

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
  logits, label_ids = eval_pred.predictions, eval_pred.label_ids
  pred_ids = np.argmax(logits, axis=-1)
  true_flat, pred_flat, valid = [], [], {"B", "I", "O"}

  for pred_row, label_row in zip(pred_ids, label_ids):
    resp_mask = label_row != -100
    gold_ids  = label_row[resp_mask]
    gold_str  = tokenizer.decode(gold_ids, skip_special_tokens=True).strip().split()
    gold_str  = [t if t in valid else "O" for t in gold_str]
    n         = len(gold_str)

    decoded = tokenizer.decode(pred_row[resp_mask],
                                skip_special_tokens=True).strip().split()
    decoded = (decoded + ["O"] * n)[:n]
    decoded = [t if t in valid else "O" for t in decoded]

    true_flat.extend(gold_str)
    pred_flat.extend(decoded)

  return {
      "accuracy": accuracy_score(true_flat, pred_flat),
      "P":  precision_score(true_flat, pred_flat, labels=["B","I"], average="macro", zero_division=0),
      "R":  recall_score(true_flat, pred_flat, labels=["B","I"], average="macro", zero_division=0),
      "F1": f1_score(true_flat, pred_flat, labels=["B","I"], average="macro", zero_division=0)}

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

lora_config = LoraConfig(
    r=8,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model_peft = get_peft_model(model, lora_config)

In [ ]:
model.config.use_cache = False

In [ ]:
epoch_metrics_log = []
class MetricsCallback(TrainerCallback):
  def on_evaluate(self, args, state, control, metrics=None, **kwargs):
    if metrics:
      epoch_metrics_log.append({
          "epoch":    state.epoch,
          "accuracy": metrics.get("eval_accuracy", 0),
          "P":        metrics.get("eval_P", 0),
          "R":        metrics.get("eval_R", 0),
          "F1":       metrics.get("eval_F1", 0),
          "loss":     metrics.get("eval_loss", 0),
      })

In [ ]:
from transformers import DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./qwen_bio_lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-4,
    fp16=True,
    logging_strategy="epoch",
    save_strategy="epoch",
    eval_strategy="epoch",
    eval_steps=20,
    gradient_accumulation_steps=16,
    report_to="none",
    metric_for_best_model="F1"
)

trainer = Trainer(
    model=model_peft,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer, padding=True),
    compute_metrics=compute_metrics,
    callbacks=[MetricsCallback()]
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.eval()

true_labels, pred_labels = [], []
for sample in tqdm(eval_data):
  pred = predict_with_qwen(model, sample)
  true_labels.extend(sample["labels"])
  pred_labels.extend(pred)

In [ ]:
finetuned_metrics = {
    "accuracy": accuracy_score(true_labels, pred_labels),
    "P": precision_score(true_labels, pred_labels, average="macro", zero_division=0),
    "R":  recall_score(true_labels, pred_labels, average="macro", zero_division=0),
    "F1": f1_score(true_labels, pred_labels, average="macro", zero_division=0)}

print("EVALUATION")
print(classification_report(true_labels, pred_labels, labels=["B", "I", "O"], zero_division=0))

print(
    f"accuracy {finetuned_metrics['accuracy'] * 100:.0f}%, "
    f"F1 {finetuned_metrics['F1'] * 100:.0f}%, "
    f"P {finetuned_metrics['P'] * 100:.0f}%, "
    f"R {finetuned_metrics['R'] * 100:.0f}%"
)